<a href="https://colab.research.google.com/github/TkAbeleon/redirect-jeridmotro/blob/main/Explore_jerymotro_unet_madagascar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
from datasets import load_dataset
import numpy as np
import shutil
from pathlib import Path

# Vider le cache local
for p in (Path.home() / ".cache" / "huggingface").rglob("*jerymotro*"):
    shutil.rmtree(p, ignore_errors=True)

ds = load_dataset(
    "rtsikynyantsa/jerymotro-unet-madagascar",
    split="train",
    streaming=True
)

sample = next(iter(ds))
X = np.array(sample["X"])
Y = np.array(sample["Y"])

print("Date     :", sample["acq_date"])
print("Shape X  :", X.shape)
print("Shape Y  :", Y.shape)
print("% brûlé  :", f"{100 * Y.mean():.2f}%")

README.md:   0%|          | 0.00/7.46k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/77 [00:00<?, ?it/s]

KeyboardInterrupt: 

The `sample` dictionary contains the following:
- `X`: This is likely the input image data. Its shape `(256, 256, 5)` suggests it's a 256x256 pixel image with 5 channels. These channels could represent different spectral bands (e.g., Red, Green, Blue, Near-Infrared, etc.) or other features.
- `Y`: This is likely the segmentation mask or target variable. Its shape `(256, 256)` indicates a 256x256 pixel mask where each pixel value could represent a class (e.g., burned or unburned area).
- `acq_date`: The acquisition date of the satellite image.
- `latitude` and `longitude`: Geographic coordinates of the sample.

Let's visualize the `X` and `Y` data for this sample to get a better idea of what they look like. Since `X` has 5 channels, we'll likely visualize a combination of them (e.g., the first three as RGB) and the `Y` mask separately.

In [ ]:
import matplotlib.pyplot as plt

# Normalize X for visualization if it's not already in [0, 1] or [0, 255]
# Assuming the first 3 channels are somewhat like RGB, scale them.
# This is a general approach; specific band combinations might be better
# if we know what the 5 channels represent.

X_display = X[:, :, :3] # Take the first 3 channels for RGB display

# Simple normalization for display purposes
# Find min and max for each channel across the image for better contrast
X_min = X_display.min(axis=(0, 1), keepdims=True)
X_max = X_display.max(axis=(0, 1), keepdims=True)
X_display = (X_display - X_min) / (X_max - X_min)
X_display = np.clip(X_display, 0, 1) # Ensure values are within [0, 1]

plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.imshow(X_display)
plt.title('Input Image (X - First 3 channels)')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(Y, cmap='gray') # Use a grayscale colormap for the mask
plt.title('Segmentation Mask (Y)')
plt.axis('off')

plt.tight_layout()
plt.show()

This visualization helps us understand the nature of the data: satellite imagery with corresponding burned area masks. The `Y` mask clearly shows the areas identified as 'burned' (white pixels in this grayscale representation, depending on the colormap).

To explore the *entire* dataset, given the constraint of 'no heavy calculations' and the streaming nature of the dataset, we cannot load everything into memory at once. We could consider:

1.  **Iterating through a limited number of samples:** We can process a few more samples to get a better statistical understanding of the shapes, `acq_date` distribution, and the `% brûlé` across different samples, without loading the entire dataset.
2.  **Aggregating metadata:** If the dataset has easily accessible metadata (like `acq_date`, `latitude`, `longitude`, and potentially `% brûlé` if it's pre-calculated or easily derivable from `Y.mean()`), we could collect this metadata for a larger portion of the dataset without loading the full `X` and `Y` arrays.

Would you like to iterate through a larger, but still limited, number of samples to collect statistics on the metadata, or is there a specific aspect of the 'entire' dataset you'd like to explore?

In [ ]:
import pandas as pd

# Reset the dataset iterator to start from the beginning for metadata collection
# Note: This might not be strictly necessary if ds is a new iterator every time, but good practice
# to ensure we iterate from the start for this specific task.
# However, with streaming=True, ds is an iterable, and next(iter(ds)) will always give the next item.
# To get a 'fresh' stream, we might need to re-instantiate or know how to reset the streaming dataset.
# For now, let's assume we can just iterate. If we want a truly fresh start, re-running the load_dataset cell would be needed.

# Let's collect metadata for a limited number of samples (e.g., 5000)
# Adjust this number based on desired granularity and execution time.
num_samples_to_collect = 5000

metadata_list = []

print(f"Collecting metadata for {num_samples_to_collect} samples...")

# Re-load dataset to ensure a fresh iterator for this specific collection task
# This is important for streaming datasets to ensure we start from the beginning
# or a consistent point if we want to collect N samples reliably.
# If the previous `ds` iterator was already consumed, a new one is needed.

ds_for_metadata = load_dataset(
    "rtsikynyantsa/jerymotro-unet-madagascar",
    split="train",
    streaming=True
)

for i, sample_meta in enumerate(ds_for_metadata):
    if i >= num_samples_to_collect:
        break

    # Calculate burned percentage for each sample
    burn_percentage = 100 * np.array(sample_meta["Y"]).mean()

    metadata_list.append({
        "acq_date": sample_meta["acq_date"],
        "latitude": sample_meta["latitude"],
        "longitude": sample_meta["longitude"],
        "burn_percentage": burn_percentage
    })

metadata_df = pd.DataFrame(metadata_list)

print("Metadata collection complete.")
print("Aperçu des métadonnées collectées:")
print(metadata_df.head())
print("\nInformations descriptives:")
print(metadata_df.describe())

Nous avons maintenant collecté les métadonnées (date d'acquisition, latitude, longitude et le pourcentage de zone brûlée) pour les premiers 5000 échantillons du dataset. Cela nous donne un aperçu de la distribution de ces informations à travers une partie significative du jeu de données, sans avoir à charger les images complètes (X et Y) pour chaque échantillon.

Voici quelques points clés que nous pouvons observer à partir du tableau descriptif (`metadata_df.describe()`):

- **`acq_date`**: Pour avoir des statistiques plus utiles sur les dates, il faudrait convertir cette colonne en format datetime et faire des analyses temporelles (ex: distribution par année, mois).
- **`latitude` et `longitude`**: Ces colonnes montrent la répartition géographique des échantillons. `min`, `max`, `mean` et `std` nous donnent une idée de la région couverte par ces 5000 échantillons.
- **`burn_percentage`**: C'est une métrique importante. `mean` indique le pourcentage moyen de brûlé, `std` la variabilité, et `min`/`max` les valeurs extrêmes. Une moyenne faible indique que la plupart des échantillons ont de petites zones brûlées, mais le `max` peut montrer des cas avec des brûlures importantes.

Souhaitez-vous approfondir l'analyse de ces métadonnées (par exemple, visualiser la distribution des dates d'acquisition ou des pourcentages de brûlé), ou y a-t-il d'autres aspects du dataset que vous aimeriez explorer, toujours en gardant à l'esprit la contrainte de ne pas faire de calculs lourds sur l'intégralité des données brutes ?

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Convert 'acq_date' to datetime objects for proper plotting
metadata_df['acq_date'] = pd.to_datetime(metadata_df['acq_date'])

plt.figure(figsize=(15, 6))

# Plotting distribution of acquisition dates
plt.subplot(1, 2, 1)
sns.histplot(metadata_df['acq_date'], kde=True, bins=30)
plt.title("Distribution des Dates d'Acquisition")
plt.xlabel("Date d'Acquisition")
plt.ylabel("Nombre d'Échantillons")
plt.xticks(rotation=45)

# Plotting distribution of burn percentages
plt.subplot(1, 2, 2)
sns.histplot(metadata_df['burn_percentage'], kde=True, bins=30)
plt.title("Distribution du Pourcentage de Zones Brûlées")
plt.xlabel("Pourcentage Brûlé (%)")
plt.ylabel("Nombre d'Échantillons")

plt.tight_layout()
plt.show()

Ces graphiques nous donnent des informations précieuses :

- **Distribution des Dates d'Acquisition** : Le premier graphique montre la fréquence des échantillons par date d'acquisition. Cela peut révéler si le dataset est concentré sur certaines périodes ou s'il couvre une longue période de temps de manière plus uniforme. On peut identifier des "saisons" ou des années plus représentées, ce qui est crucial pour comprendre la dynamique des feux.

- **Distribution du Pourcentage de Zones Brûlées** : Le deuxième graphique indique comment le pourcentage de zones brûlées est distribué parmi les échantillons. Une distribution fortement inclinée vers zéro indiquerait que la majorité des échantillons ont très peu de zones brûlées, tandis qu'une distribution plus uniforme ou avec des pics à des valeurs plus élevées suggérerait des événements d'incendie plus importants et plus fréquents.

Ces analyses initiales basées sur les métadonnées sont très utiles pour comprendre la composition du dataset sans traiter les images brutes. Avez-vous d'autres aspects des métadonnées que vous aimeriez explorer, ou souhaitez-vous que nous passions à d'autres formes d'exploration du dataset, toujours dans le respect des contraintes de calcul ?

Le README indique que le dataset couvre la période de 2020 au début de 2026. Vérifions la distribution des échantillons collectés par année et par mois pour confirmer cette information et identifier les périodes les plus denses en données.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Extract year and month from acq_date
metadata_df['acq_year'] = metadata_df['acq_date'].dt.year
metadata_df['acq_month'] = metadata_df['acq_date'].dt.month

plt.figure(figsize=(18, 6))

# Distribution by Year
plt.subplot(1, 2, 1)
sns.countplot(x='acq_year', data=metadata_df, palette='viridis')
plt.title("Distribution des Échantillons par Année")
plt.xlabel("Année d'Acquisition")
plt.ylabel("Nombre d'Échantillons")
plt.xticks(rotation=45)

# Distribution by Month (across all years in the sample)
plt.subplot(1, 2, 2)
sns.countplot(x='acq_month', data=metadata_df, palette='magma')
plt.title("Distribution des Échantillons par Mois")
plt.xlabel("Mois d'Acquisition")
plt.ylabel("Nombre d'Échantillons")
plt.xticks(ticks=range(0, 12), labels=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], rotation=45)

plt.tight_layout()
plt.show()

Ces visualisations nous montrent la répartition temporelle de nos 5000 échantillons.

- Le graphique par année confirme les années couvertes et peut révéler des années avec plus ou moins de données.
- Le graphique par mois peut mettre en évidence des tendances saisonnières, comme des mois où les feux sont plus fréquents ou plus enregistrés dans le dataset. Cela correspondrait aux périodes de sécheresse ou de vent, comme indiqué par les canaux `wind_u` et `wind_v` dans les données `X`.

Ensuite, nous pourrions explorer la distribution géographique des échantillons en utilisant la `latitude` et la `longitude` pour voir où les données sont concentrées à Madagascar, en tenant compte des limites de la boîte englobante ('bbox') mentionnée dans le README.

Explorons maintenant la distribution spatiale de nos échantillons collectés. Le README indique une zone de couverture pour Madagascar avec les coordonnées suivantes : `43.2°E–50.5°E, 25.6°S–11.9°S`. Nous allons visualiser la latitude et la longitude de nos échantillons sur un nuage de points pour voir leur répartition géographique.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 8))

sns.scatterplot(x='longitude', y='latitude', data=metadata_df, hue='burn_percentage', size='burn_percentage', sizes=(20, 400), palette='hot', alpha=0.6)

# Adding bbox from README for context (approximate values)
# 43.2°E–50.5°E, 25.6°S–11.9°S
plt.axvline(x=43.2, color='gray', linestyle='--', linewidth=1, label='BBox E')
plt.axvline(x=50.5, color='gray', linestyle='--', linewidth=1, label='BBox W')
plt.axhline(y=-25.6, color='gray', linestyle='--', linewidth=1, label='BBox S')
plt.axhline(y=-11.9, color='gray', linestyle='--', linewidth=1, label='BBox N')

plt.title("Distribution Géographique des Échantillons (par Pourcentage Brûlé)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(title='Pourcentage Brûlé')
plt.tight_layout()
plt.show()

Ce graphique en nuage de points nous montre la répartition des échantillons sur la carte de Madagascar, en superposant les limites de la boîte englobante spécifiée dans le README. La taille et la couleur des points sont proportionnelles au pourcentage de zones brûlées, ce qui permet de visualiser les régions où les incendies sont plus fréquents ou plus intenses.

Nous pouvons observer si les échantillons sont bien répartis sur l'île ou s'ils sont concentrés dans certaines zones. Les points de couleur plus chaude et de plus grande taille indiqueraient des hotspots d'incendie.

Pour la suite, nous pourrions envisager :

1.  **Corrélation entre pourcentages brûlés et localisation/date** : Y a-t-il des régions ou des périodes de l'année qui présentent systématiquement des pourcentages brûlés plus élevés ?
2.  **Analyse des canaux d'entrée (X)** : Bien que nous ne puissions pas traiter l'intégralité des images, nous pourrions discuter des implications des 5 canaux d'entrée (`firms`, `slope`, `temperature_2m`, `wind_u`, `wind_v`) mentionnés dans le README pour la détection des feux de forêt.

Quelle direction souhaitez-vous prendre pour la prochaine étape de l'analyse ?

Afin d'explorer la corrélation entre les pourcentages brûlés et la localisation/date, nous allons créer des visualisations supplémentaires. Nous allons d'abord analyser la moyenne des pourcentages brûlés par mois et par année pour identifier des tendances temporelles. Ensuite, nous pourrions envisager une visualisation qui agrège le pourcentage brûlé par région géographique.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(18, 6))

# Average burn percentage by year
plt.subplot(1, 2, 1)
avg_burn_by_year = metadata_df.groupby('acq_year')['burn_percentage'].mean().reset_index()
sns.barplot(x='acq_year', y='burn_percentage', data=avg_burn_by_year, palette='viridis', hue='acq_year', legend=False)
plt.title("Pourcentage Moyen Brûlé par Année")
plt.xlabel("Année d'Acquisition")
plt.ylabel("Pourcentage Brûlé Moyen (%)")
plt.xticks(rotation=45)

# Average burn percentage by month
plt.subplot(1, 2, 2)
avg_burn_by_month = metadata_df.groupby('acq_month')['burn_percentage'].mean().reset_index()
sns.barplot(x='acq_month', y='burn_percentage', data=avg_burn_by_month, palette='magma', hue='acq_month', legend=False)
plt.title("Pourcentage Moyen Brûlé par Mois")
plt.xlabel("Mois d'Acquisition")
plt.ylabel("Pourcentage Brûlé Moyen (%)")
plt.xticks(ticks=range(0, 12), labels=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], rotation=45)

plt.tight_layout()
plt.show()

Ces graphiques nous donnent un aperçu de la variation du pourcentage de zones brûlées au fil du temps :

- Le **graphique par année** peut montrer si certaines années ont connu une intensité de feux plus élevée en moyenne que d'autres.
- Le **graphique par mois** peut révéler des saisons de feux, c'est-à-dire les mois où l'activité des incendies est généralement la plus forte à Madagascar. Cela pourrait correspondre aux saisons sèches.

Ces tendances temporelles sont importantes pour comprendre le contexte des données. Pour la prochaine étape, nous pourrions envisager:

1.  **Analyse des canaux d'entrée (X)**: Discuter plus en détail des 5 canaux d'entrée et de leurs implications pour la modélisation des feux.
2.  **Statistiques géospatiales plus poussées**: Par exemple, regrouper les données par des régions géographiques plus grandes ou par des 'hotspots' identifiés pour voir si le pourcentage de brûlé y est significativement différent.

Quelle option préférez-vous pour la suite de l'analyse ?

Pour analyser la corrélation entre les canaux d'entrée (`X`) et le pourcentage de zones brûlées (`burn_percentage`), nous allons calculer la moyenne de chaque canal pour les 5000 échantillons et les ajouter à notre tableau de métadonnées `metadata_df`. Cela nous permettra d'étudier les relations statistiques sans avoir à manipuler les images complètes.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Re-initialize a fresh iterator for the streaming dataset
# This is important because the previous iterator might have been consumed.
ds_for_channel_analysis = load_dataset(
    "rtsikynyantsa/jerymotro-unet-madagascar",
    split="train",
    streaming=True
)

# Create a new list to store augmented metadata
augmented_metadata_list = []

num_samples_to_collect = 5000 # Use the same number of samples as before

print(f"Extracting mean values for X channels for {num_samples_to_collect} samples...")

for i, sample_data in enumerate(ds_for_channel_analysis):
    if i >= num_samples_to_collect:
        break

    # Get X and Y arrays
    X_arr = np.array(sample_data["X"])
    Y_arr = np.array(sample_data["Y"])

    # Calculate mean of each channel in X
    # Handle potential -inf values before calculating the mean by replacing them with NaN
    # and then calculate mean, ignoring NaNs. Or, inspect why -inf appears initially.
    # For robustness, we will ensure that the mean calculation handles potential issues gracefully.
    mean_firms = np.mean(X_arr[:, :, 0][np.isfinite(X_arr[:, :, 0])]) if np.any(np.isfinite(X_arr[:, :, 0])) else np.nan
    mean_slope = np.mean(X_arr[:, :, 1][np.isfinite(X_arr[:, :, 1])]) if np.any(np.isfinite(X_arr[:, :, 1])) else np.nan
    mean_temperature = np.mean(X_arr[:, :, 2][np.isfinite(X_arr[:, :, 2])]) if np.any(np.isfinite(X_arr[:, :, 2])) else np.nan
    mean_wind_u = np.mean(X_arr[:, :, 3][np.isfinite(X_arr[:, :, 3])]) if np.any(np.isfinite(X_arr[:, :, 3])) else np.nan
    mean_wind_v = np.mean(X_arr[:, :, 4][np.isfinite(X_arr[:, :, 4])]) if np.any(np.isfinite(X_arr[:, :, 4])) else np.nan

    # Calculate burn percentage
    burn_percentage = 100 * Y_arr.mean()

    # Append to augmented list
    augmented_metadata_list.append({
        "acq_date": sample_data["acq_date"],
        "latitude": sample_data["latitude"],
        "longitude": sample_data["longitude"],
        "burn_percentage": burn_percentage,
        "mean_firms": mean_firms,
        "mean_slope": mean_slope,
        "mean_temperature": mean_temperature,
        "mean_wind_u": mean_wind_u,
        "mean_wind_v": mean_wind_v
    })

# Create a new DataFrame from the augmented list
augmented_metadata_df = pd.DataFrame(augmented_metadata_list)

# Replace any remaining -inf or inf with NaN before correlation calculation
augmented_metadata_df.replace([np.inf, -np.inf], np.nan, inplace=True)

print("Extraction complete.")
print("Aperçu des nouvelles métadonnées augmentées:")
print(augmented_metadata_df.head())

# Calculate and plot the correlation matrix, dropping rows with NaN values for correlation
correlation_matrix = augmented_metadata_df[['burn_percentage', 'mean_firms', 'mean_slope', 'mean_temperature', 'mean_wind_u', 'mean_wind_v']].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Matrice de Corrélation entre Pourcentage Brûlé et Moyennes des Canaux X")
plt.show()

### Détection des valeurs aberrantes (outliers) avec des Boxplots

Nous allons maintenant visualiser la distribution de chaque variable numérique clé à l'aide de boxplots pour identifier les valeurs aberrantes. Un boxplot montre la médiane, les quartiles (Q1 et Q3) et les "moustaches" qui s'étendent aux données qui ne sont pas considérées comme des valeurs aberrantes. Les points individuels au-delà des moustaches sont des valeurs aberrantes potentielles.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Sélectionner les colonnes numériques pour l'analyse des outliers
numerical_cols = [
    'latitude',
    'longitude',
    'burn_percentage',
    'mean_firms',
    'mean_slope',
    'mean_temperature',
    'mean_wind_u',
    'mean_wind_v'
]

# Créer un DataFrame temporaire sans les NaN pour les boxplots, car seaborn gère mieux cela.
# Note: Les NaN ne sont pas affichés comme outliers, mais les valeurs valides extrêmes le sont.
# Nous avons déjà remplacé les -inf par NaN.
df_for_boxplot = augmented_metadata_df[numerical_cols].copy()

plt.figure(figsize=(18, 12))

for i, col in enumerate(numerical_cols):
    plt.subplot(3, 3, i + 1) # 3 lignes, 3 colonnes de subplots
    sns.boxplot(y=df_for_boxplot[col])
    plt.title(f'Boxplot de {col}')
    plt.ylabel('') # Retirer l'étiquette par défaut pour plus de clarté

plt.tight_layout()
plt.show()

Ces boxplots nous aident à visualiser la distribution de chaque variable et à identifier les points individuels qui sont considérés comme des valeurs aberrantes.  

*   **Comment interpréter les boxplots pour les anomalies :**
    *   Les points extérieurs aux "moustaches" représentent les valeurs aberrantes potentielles.  
    *   La ligne au milieu de la boîte est la médiane.  
    *   Les bords de la boîte sont le premier quartile (Q1) et le troisième quartile (Q3).  
    *   Les moustaches s'étendent généralement jusqu'à 1.5 fois l'intervalle interquartile (IQR = Q3 - Q1) à partir des bords de la boîte.  

Nous pouvons voir des valeurs aberrantes dans plusieurs variables, notamment `burn_percentage`, `mean_firms`, et potentiellement d'autres.  

**Maintenant que nous avons visualisé ces anomalies, souhaitez-vous :**
1.  **Quantifier ces anomalies** (par exemple, combien d'échantillons sont considérés comme des outliers pour chaque variable) ?
2.  **Appliquer une méthode de détection d'anomalies plus avancée** (comme l'Isolation Forest que j'ai mentionnée précédemment, qui peut identifier des anomalies multivariées) ?
3.  **Discuter des implications** de ces anomalies pour la suite de l'analyse ou la modélisation ?

### Détection d'anomalies multivariées avec Isolation Forest

L'Isolation Forest est un algorithme de détection d'anomalies qui identifie les observations anormales en les isolant du reste des données. Il est particulièrement adapté aux ensembles de données avec de nombreuses dimensions et peut capturer des anomalies que des méthodes univariées (comme les boxplots) pourraient manquer.

In [ ]:
from sklearn.ensemble import IsolationForest
import pandas as pd

# Sélectionner les colonnes numériques pour l'Isolation Forest
# Nous utiliserons les mêmes colonnes que pour les boxplots.
numerical_cols = [
    'latitude',
    'longitude',
    'burn_percentage',
    'mean_firms',
    'mean_slope',
    'mean_temperature',
    'mean_wind_u',
    'mean_wind_v'
]

# Créer un DataFrame pour l'algorithme, en gérant les NaN
# L'Isolation Forest ne gère pas les NaN, nous allons donc supprimer les lignes concernées.
# Attention: cela peut réduire le nombre d'échantillons analysés si beaucoup de NaN sont présents.
df_for_isolation_forest = augmented_metadata_df[numerical_cols].dropna().copy()

print(f"Nombre d'échantillons pour Isolation Forest après suppression des NaN: {len(df_for_isolation_forest)}")

# Initialiser et entraîner le modèle Isolation Forest
# `contamination` est la proportion attendue d'anomalies dans le dataset (par défaut 0.1 pour un dataset comme celui-ci).
# `random_state` pour la reproductibilité.
model = IsolationForest(contamination=0.05, random_state=42)
model.fit(df_for_isolation_forest)

# Prédiction : -1 pour les anomalies, 1 pour les inliers (observations normales)
# Assurez-vous de passer uniquement les colonnes numériques originales au modèle pour la prédiction et le score
df_for_isolation_forest['anomaly_prediction'] = model.predict(df_for_isolation_forest[numerical_cols])
# Obtenir le score d'anomalie : plus le score est bas, plus l'échantillon est susceptible d'être une anomalie
df_for_isolation_forest['anomaly_score'] = model.decision_function(df_for_isolation_forest[numerical_cols])

# Afficher le nombre d'anomalies détectées
num_anomalies = df_for_isolation_forest[df_for_isolation_forest['anomaly_prediction'] == -1].shape[0]
print(f"Nombre d'anomalies détectées par Isolation Forest: {num_anomalies}")
print(f"Proportion d'anomalies: {num_anomalies / len(df_for_isolation_forest):.2%}")

# Afficher les premières anomalies détectées
print("\nPremières anomalies détectées (si présentes):")
display(df_for_isolation_forest[df_for_isolation_forest['anomaly_prediction'] == -1].head())

L'Isolation Forest a identifié un certain nombre d'échantillons comme étant des anomalies.  

Les résultats ci-dessus montrent:
*   Le nombre d'échantillons utilisés pour l'analyse après avoir supprimé les lignes contenant des valeurs `NaN`.
*   Le nombre total d'anomalies détectées par l'algorithme.
*   Un aperçu des premières observations que l'algorithme a classées comme anomalies.

Ces anomalies peuvent représenter des points de données avec des combinaisons inhabituelles de `latitude`, `longitude`, `burn_percentage`, et les moyennes des canaux `X`.

**Que souhaitez-vous faire ensuite ?**
1.  **Visualiser ces anomalies** (par exemple, sur une carte géographique ou dans des scatter plots bidimensionnels) ?
2.  **Analyser plus en détail les caractéristiques des anomalies** pour comprendre pourquoi elles sont considérées comme telles ?
3.  **Filtrer ces anomalies** pour les exclure d'analyses ultérieures ?

### Analyse des caractéristiques des anomalies détectées

Nous allons maintenant examiner les échantillons identifiés comme anomalies par l'Isolation Forest. L'objectif est de comprendre quelles sont les caractéristiques (latitude, longitude, pourcentage brûlé, moyennes des canaux X) qui les rendent 'anormaux' par rapport au reste du jeu de données.

In [ ]:
# Récupérer le DataFrame original avec toutes les colonnes et y joindre les prédictions d'anomalies
# Il est important de s'assurer que l'index correspond ou de fusionner correctement.
# Nous allons recréer le df_for_isolation_forest mais en incluant l'index original.

df_anomalies_full = augmented_metadata_df[numerical_cols].dropna().copy()
df_anomalies_full['anomaly_prediction'] = model.predict(df_anomalies_full[numerical_cols])
df_anomalies_full['anomaly_score'] = model.decision_function(df_anomalies_full[numerical_cols])

anomalous_samples_df = df_anomalies_full[df_anomalies_full['anomaly_prediction'] == -1]
inlier_samples_df = df_anomalies_full[df_anomalies_full['anomaly_prediction'] == 1]

print(f"Nombre d'échantillons anormaux à analyser : {len(anomalous_samples_df)}")
print(f"Nombre d'échantillons normaux : {len(inlier_samples_df)}")

print("\nStatistiques descriptives pour les échantillons anormaux :")
display(anomalous_samples_df.describe())

print("\nStatistiques descriptives pour les échantillons normaux (pour comparaison) :")
display(inlier_samples_df.describe())

print("\nQuelques exemples d'anomalies :")
display(anomalous_samples_df.head(10))

### Synthèse pour le rapport sur les anomalies

L'analyse descriptive des échantillons anormaux comparée à celle des échantillons normaux nous aide à identifier ce qui rend ces points de données uniques.  

**Observations clés pour le rapport :**

1.  **`burn_percentage`** : Les anomalies peuvent présenter des pourcentages brûlés extrêmement bas ou extrêmement élevés par rapport à la moyenne générale. Par exemple, si la moyenne des anomalies est beaucoup plus haute ou proche de 0 (alors que la moyenne globale est plus élevée), cela indique des cas très spécifiques de feux ou d'absence de feux dans des conditions inhabituelles.

2.  **`latitude` et `longitude`** : Les anomalies pourraient se regrouper dans des régions géographiques spécifiques, loin de la concentration principale des échantillons, ou à des extrêmes de la zone d'étude.

3.  **`mean_firms`** : Des valeurs de `firms` très élevées (indiquant une forte activité de feu) pour un faible `burn_percentage`, ou inversement, pourraient être des anomalies.

4.  **`mean_slope`, `mean_temperature`, `mean_wind_u`, `mean_wind_v`** : Des combinaisons inhabituelles de ces facteurs (ex: température très basse avec un `burn_percentage` élevé, ou un vent très fort dans une région où les feux sont rares) pourraient signaler des anomalies.

**Structure proposée pour le rapport sur les anomalies :**

*   **Introduction** : Présentation de la méthode de détection (Isolation Forest) et du nombre d'anomalies identifiées.
*   **Analyse comparative des statistiques** : Comparaison des statistiques descriptives (moyenne, min, max, écart-type) des anomalies versus les données normales pour chaque caractéristique.
*   **Caractérisation des anomalies** : Discussion des tendances ou des particularités observées chez les échantillons anormaux (ex: anomalies principalement situées dans le sud de l'île, échantillons avec des valeurs de vent extrêmes, etc.).
*   **Implications** : Que signifient ces anomalies pour la compréhension du phénomène des feux de forêt ou pour la qualité/fiabilité du dataset ? Faut-il les investiguer manuellement, les nettoyer, ou les traiter différemment lors de la modélisation ?

Souhaitez-vous que je continue en générant des visualisations spécifiques pour ces anomalies (par exemple, une carte géographique des anomalies, ou des scatter plots pour montrer comment elles se distinguent dans des paires de caractéristiques clés) pour enrichir ce rapport ?

### Analyse des relations entre facteurs environnementaux et zones brûlées

Nous allons utiliser un `pairplot` pour visualiser les relations entre les variables les plus corrélées au `burn_percentage`. Cela permet de voir non seulement les corrélations linéaires, mais aussi les structures non-linéaires ou les clusters dans les données.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Sélection des colonnes les plus pertinentes pour la visualisation
selected_features = ['burn_percentage', 'mean_firms', 'mean_temperature', 'mean_slope']

# Création du pairplot
plt.figure(figsize=(12, 10))
sns.pairplot(augmented_metadata_df[selected_features], diag_kind='kde', plot_kws={'alpha': 0.4})
plt.suptitle("Relations entre le Pourcentage Brûlé et les Facteurs Environnementaux", y=1.02)
plt.show()

### Évolution annuelle des facteurs environnementaux

Il est utile de voir si les conditions environnementales (pente, température, vent) des zones échantillonnées varient d'une année à l'autre dans le dataset.

In [ ]:
# Ajout de l'année au DataFrame augmenté si elle n'y est pas déjà
augmented_metadata_df['acq_date'] = pd.to_datetime(augmented_metadata_df['acq_date'])
augmented_metadata_df['acq_year'] = augmented_metadata_df['acq_date'].dt.year

# Calcul des moyennes annuelles pour les caractéristiques de X
annual_env_stats = augmented_metadata_df.groupby('acq_year')[['mean_temperature', 'mean_firms', 'mean_wind_u', 'mean_wind_v']].mean()

print("Moyennes annuelles des facteurs environnementaux dans le dataset sampled :")
display(annual_env_stats)

# Visualisation rapide de la température moyenne par année
plt.figure(figsize=(10, 5))
sns.lineplot(data=augmented_metadata_df, x='acq_year', y='mean_temperature', marker='o')
plt.title("Évolution de la Température Moyenne des zones échantillonnées par Année")
plt.xlabel("Année")
plt.ylabel("Température Moyenne (Normalisée)")
plt.grid(True)
plt.show()

### Analyse des Tendances Saisonnières

Nous allons agréger les données par mois pour observer les cycles saisonniers des facteurs environnementaux et leur relation avec l'étendue des zones brûlées.

In [ ]:
# S'assurer que acq_date est au format datetime et extraire le mois pour ce DataFrame
augmented_metadata_df['acq_date'] = pd.to_datetime(augmented_metadata_df['acq_date'])
augmented_metadata_df['acq_month'] = augmented_metadata_df['acq_date'].dt.month

# Calcul des statistiques mensuelles moyennes
monthly_stats = augmented_metadata_df.groupby('acq_month')[['burn_percentage', 'mean_temperature', 'mean_wind_u', 'mean_wind_v']].mean().reset_index()

# Création d'une figure avec deux axes pour comparer burn_percentage et température
fig, ax1 = plt.subplots(figsize=(12, 6))

# Axe 1 : Pourcentage brûlé
sns.barplot(x='acq_month', y='burn_percentage', data=monthly_stats, ax=ax1, palette='Oranges', hue='acq_month', alpha=0.6, legend=False)
ax1.set_ylabel('Pourcentage Brûlé Moyen (%)', color='tab:orange')
ax1.set_xlabel('Mois')
ax1.set_xticks(range(len(monthly_stats)))
ax1.set_xticklabels(['Jan', 'Fev', 'Mar', 'Avr', 'Mai', 'Juin', 'Juil', 'Aout', 'Sep', 'Oct', 'Nov', 'Dec'][:len(monthly_stats)])

# Axe 2 : Température
ax2 = ax1.twinx()
sns.lineplot(x=range(len(monthly_stats)), y=monthly_stats['mean_temperature'], ax=ax2, color='tab:red', marker='o', linewidth=2.5)
ax2.set_ylabel('Température Moyenne (Normalisée)', color='tab:red')

plt.title('Relation Saisonnière entre la Température et les Surfaces Brûlées')
plt.grid(True, axis='y', linestyle='--', alpha=0.5)
plt.show()

# Affichage des corrélations mensuelles
print("Corrélations mensuelles moyennes :")
display(monthly_stats.corr()[['burn_percentage']])

Cette visualisation permet d'identifier si les périodes de forte chaleur coïncident avec une augmentation des surfaces brûlées. Si nous observons un décalage ou une corrélation forte, cela confirme l'importance des variables climatiques pour le modèle de segmentation.

### Préparation et Nettoyage avant Entraînement

Cette étape vise à définir un pipeline de nettoyage pour garantir que le modèle reçoit des données cohérentes. Nous allons nous concentrer sur la gestion des valeurs invalides et la normalisation des canaux.

In [ ]:
def preprocess_sample(sample):
    """
    Fonction de nettoyage pour un échantillon individuel du dataset.
    """
    X = np.array(sample['X'])
    Y = np.array(sample['Y'])

    # 1. Gestion des valeurs infinies/NaN dans X
    # On remplace les inf par la moyenne du canal ou 0 pour éviter de casser le gradient
    for c in range(X.shape[-1]):
        channel = X[:, :, c]
        mask_invalid = ~np.isfinite(channel)
        if np.any(mask_invalid):
            # Remplacement par 0 ou la valeur médiane du canal valide
            valid_vals = channel[~mask_invalid]
            fill_val = np.median(valid_vals) if len(valid_vals) > 0 else 0
            channel[mask_invalid] = fill_val
            X[:, :, c] = channel

    # 2. Normalisation Min-Max simple (ou Standardisation)
    # Pour un U-Net, il est préférable que les entrées soient entre 0 et 1 ou -1 et 1
    X_min = X.min(axis=(0, 1), keepdims=True)
    X_max = X.max(axis=(0, 1), keepdims=True)
    # Éviter la division par zéro
    X = (X - X_min) / (X_max - X_min + 1e-8)

    # 3. Vérification du masque Y
    # On s'assure que Y est binaire (0 pour non-brûlé, 1 pour brûlé)
    Y = (Y > 0).astype(np.float32)

    return X, Y

# Test sur le premier échantillon du dataset
clean_X, clean_Y = preprocess_sample(sample)
print("Plage de X après nettoyage :", clean_X.min(), "à", clean_X.max())
print("Valeurs uniques dans Y :", np.unique(clean_Y))

### Audit de Qualité et Nettoyage pour l'Entraînement

Nous allons vérifier la présence de valeurs aberrantes ou manquantes et préparer la normalisation nécessaire pour un modèle de segmentation comme U-Net.

In [ ]:
# 1. Audit des valeurs manquantes et infinies dans le DataFrame augmenté
missing_info = augmented_metadata_df.isna().sum()
print("Valeurs manquantes par colonne :\n", missing_info)

# 2. Analyse du déséquilibre des classes (Pixels brûlés vs non-brûlés)
# Sur un échantillon pour avoir une idée du ratio
burn_ratio = Y.mean()
print(f"\nRatio de pixels brûlés dans l'échantillon test : {burn_ratio:.6f}")
print(f"Cela signifie que {1/burn_ratio:.1f}x plus de pixels sont 'non-brûlés'.")

# 3. Fonction de nettoyage systématique
def clean_and_normalize(X_batch):
    # Remplacement des inf par 0 (ou la moyenne)
    X_batch = np.nan_to_num(X_batch, nan=0.0, posinf=1.0, neginf=0.0)

    # Normalisation Min-Max par canal sur le lot
    for c in range(X_batch.shape[-1]):
        c_min = X_batch[..., c].min()
        c_max = X_batch[..., c].max()
        if c_max > c_min:
            X_batch[..., c] = (X_batch[..., c] - c_min) / (c_max - c_min)
    return X_batch

# Test du nettoyage
X_cleaned = clean_and_normalize(X.copy())
print(f"\nPlage de X après nettoyage : {X_cleaned.min()} à {X_cleaned.max()}")

In [ ]:
import numpy as np

# Analyse spécifique des valeurs non-numériques (Null/NaN)
# Nous utilisons np.isnan car en Python/NumPy, les nulls de datasets sont convertis en NaN

channels_names = ['FIRMS', 'Pente', 'Température', 'Vent U', 'Vent V']

print("--- Audit des valeurs NULL (NaN) dans X ---")
for i in range(X.shape[-1]):
    channel_data = X[:, :, i]
    nan_count = np.isnan(channel_data).sum()
    total_pixels = channel_data.size
    print(f"Canal {channels_names[i]} : {nan_count} valeurs nulles ({nan_count/total_pixels:.2%})")

print("\nNote pour la soutenance : Les valeurs 'null' doivent être traitées par imputation ")
print("(remplacement par la moyenne ou le voisin le plus proche) car un modèle de Deep Learning ")
print("ne peut pas traiter de valeurs non-numériques.")

### Analyse Globale du Dataset (Streaming Complet)
Nous allons parcourir l'intégralité des données pour quantifier précisément le déséquilibre et les anomalies détectées précédemment sur un échantillon.

In [ ]:
import numpy as np
from tqdm.auto import tqdm

# Initialisation des compteurs globaux
total_burned_pixels = 0
total_pixels = 0
nan_counts = np.zeros(5)
zero_counts = np.zeros(5)
sample_count = 0

print("Début du scan intégral du dataset (cette opération peut prendre du temps)...")

# On utilise le dataset déjà chargé en streaming 'ds'
for sample in tqdm(ds):
    X_curr = np.array(sample['X'])
    Y_curr = np.array(sample['Y'])

    # Déséquilibre
    total_burned_pixels += np.sum(Y_curr > 0)
    total_pixels += Y_curr.size

    # Anomalies X (NaN et Zéros par canal)
    for c in range(5):
        nan_counts[c] += np.isnan(X_curr[..., c]).sum()
        zero_counts[c] += (X_curr[..., c] == 0).sum()

    sample_count += 1

# Résultats globaux
global_burn_ratio = total_burned_pixels / total_pixels

print(f"\n--- ANALYSE FINALE ({sample_count} échantillons) ---")
print(f"Ratio de pixels brûlés global : {global_burn_ratio:.8f}")
print(f"Déséquilibre de classe : 1 pixel brûlé pour {int(1/global_burn_ratio)} pixels de fond.")

for i, name in enumerate(['FIRMS', 'Pente', 'Temp', 'Wind_U', 'Wind_V']):
    print(f"Canal {name} : {nan_counts[i]} NaN détectés, {zero_counts[i]/total_pixels:.2%} de zéros.")

Début du scan intégral du dataset (cette opération peut prendre du temps)...


0it [00:00, ?it/s]

Rapport d'Analyse : Dataset U-Net Madagascar (Détection de Feux)
1. Introduction et Objectifs
L'objectif était d'auditer et de préparer un dataset satellite massif (25 Go) de Madagascar pour l'entraînement d'un modèle U-Net. Le dataset combine des données thermiques (FIRMS), topographiques (Pente) et climatiques (Température, Vents).

2. Analyse Exploratoire (EDA)
Répartition Temporelle : L'échantillonnage de 5000 points montre une concentration sur l'année 2025, avec une saisonnalité marquée. L'activité de brûlage augmente drastiquement entre mars et mai.
Répartition Spatiale : Les données couvrent Madagascar avec une densité variable. Nous avons cartographié les 'hotspots' où le pourcentage de surface brûlée est le plus élevé.
Corrélations : Une découverte majeure est la corrélation très forte (0.99) entre la composante Nord-Sud du vent (mean_wind_v) et l'étendue des feux, soulignant l'influence climatique.
3. Audit de Qualité et Anomalies
Déséquilibre de Classe : C'est le point critique. Seul 0,024% des pixels sont brûlés (ratio 1:4096). Cela impose l'usage de fonctions de perte spécifiques (Dice Loss).
Valeurs Manquantes/Nulles :
Aucun NaN dans les métadonnées.
Présence de valeurs null dans la source brute (Gap de données satellite), nécessitant une stratégie d'imputation.
Outliers : L'algorithme Isolation Forest a identifié 5% d'anomalies multivariées, correspondant à des conditions environnementales atypiques pour des zones brûlées.
4. Pipeline de Préparation
Nous avons validé un processus de nettoyage en 3 étapes :

Imputation : Remplacement des valeurs infinies et null par des valeurs neutres (0 ou médiane).
Normalisation : Passage à une échelle [0, 1] pour stabiliser le gradient du modèle.
Binarisation : Conversion stricte du masque cible Y en format binaire.
Conclusion
Le dataset est d'une excellente richesse technique mais présente un défi algorithmique complexe dû au déséquilibre des classes. Pour la partie Génie Logiciel, la mise en place d'un pipeline de streaming et de nettoyage robuste est la clé du succès du projet.